## Creating the Dataset

The purpose of this notebook is to merge the two datasets together. Then curate the columns and set datatypes.

##### Data Sources
- Water temp data: https://library.ucsd.edu/dc/object/bb4003017c<br>
- Weather data (San Diego International Airport): https://www.ncei.noaa.gov/cdo-web/datasets/GHCND/stations/GHCND:USW00023188/detail <br>

In [1]:
import pandas as pd

In [2]:
weather_data_path = r'..\data\raw\4342131-NOAA-data-sd.csv'
pier_data_path = r'..\data\raw\pier-temp\LaJolla_TEMP_1916-202512.csv'

In [3]:
weather_df = pd.read_csv(weather_data_path)
water_df = pd.read_csv(pier_data_path,
                       header=46
                       )

In [4]:
weather_df.head(2)

,STATION,NAME,LATITUDE,LONGITUDE,ELEVATION,DATE,ACMH,ACSH,AWND,FMTM,...,WT09,WT10,WT11,WT13,WT14,WT16,WT18,WT21,WT22,WV01
0,USW00023188,"SAN DIEGO INTERNATIONAL AIRPORT, CA US",32.7336,-117.1831,4.6,1939-07-01,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,USW00023188,"SAN DIEGO INTERNATIONAL AIRPORT, CA US",32.7336,-117.1831,4.6,1939-07-02,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
water_df.head(2)

,YEAR,MONTH,DAY,TIME_PST,TIME_FLAG,SURF_TEMP_C,SURF_FLAG,BOT_TEMP_C,BOT_FLAG,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13
0,1916.0,8.0,22.0,NaN,NaN,19.5,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1916.0,8.0,23.0,NaN,NaN,19.9,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### 1. Create the date column for both dataframes

In [6]:
weather_df['date'] = pd.to_datetime(weather_df.DATE)

In [7]:
weather_df.head(2)

,STATION,NAME,LATITUDE,LONGITUDE,ELEVATION,DATE,ACMH,ACSH,AWND,FMTM,...,WT10,WT11,WT13,WT14,WT16,WT18,WT21,WT22,WV01,date
0,USW00023188,"SAN DIEGO INTERNATIONAL AIRPORT, CA US",32.7336,-117.1831,4.6,1939-07-01,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1939-07-01
1,USW00023188,"SAN DIEGO INTERNATIONAL AIRPORT, CA US",32.7336,-117.1831,4.6,1939-07-02,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1939-07-02


For the water dataframe, it became apparent that there are some completely empty rows of data. Those are removed.

In [8]:
water_df.dropna(subset='YEAR', inplace=True)

In [9]:
water_df['YEAR'] = water_df.YEAR.astype(int).astype(str)
water_df['MONTH'] = water_df.MONTH.astype(int).astype(str)
water_df['DAY'] = water_df.DAY.astype(int).astype(str)

water_df['DATE'] = water_df.YEAR + '-' + water_df.MONTH + '-' + water_df.DAY

water_df['date'] = pd.to_datetime(water_df.DATE)

In [10]:
water_df.head(2)

,YEAR,MONTH,DAY,TIME_PST,TIME_FLAG,SURF_TEMP_C,SURF_FLAG,BOT_TEMP_C,BOT_FLAG,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,DATE,date
0,1916,8,22,NaN,NaN,19.5,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1916-8-22,1916-08-22
1,1916,8,23,NaN,NaN,19.9,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1916-8-23,1916-08-23


#### 2. Remove unneeded columns

In [11]:
list(water_df.columns)

['YEAR',
 'MONTH',
 'DAY',
 'TIME_PST',
 'TIME_FLAG',
 'SURF_TEMP_C',
 'SURF_FLAG',
 'BOT_TEMP_C',
 'BOT_FLAG',
 'Unnamed: 9',
 'Unnamed: 10',
 'Unnamed: 11',
 'Unnamed: 12',
 'Unnamed: 13',
 'DATE',
 'date']

In [12]:
drop_water_cols = ['Unnamed: 9',
 'Unnamed: 10',
 'Unnamed: 11',
 'Unnamed: 12',
 'Unnamed: 13',
 'DATE']
water_df.drop(columns=drop_water_cols, inplace=True)

In [13]:
water_df.head(2)

,YEAR,MONTH,DAY,TIME_PST,TIME_FLAG,SURF_TEMP_C,SURF_FLAG,BOT_TEMP_C,BOT_FLAG,date
0,1916,8,22,NaN,NaN,19.5,0.0,NaN,NaN,1916-08-22
1,1916,8,23,NaN,NaN,19.9,0.0,NaN,NaN,1916-08-23


To ensure every date is accounted for, a base_df is created with every date from the minimum available weather date, to the maximum available water temperature date.

In [ ]:
start_date = weather_df.date.min()
end_date = water_df.date.max().date()

# create the continuous index
date_index = pd.date_range(start=start_date, end=end_date)

In [15]:
base_df = pd.DataFrame(
    {'date': list(date_index)}
)

In [16]:
merge_1_df = base_df.merge(
    weather_df,
    how='left',
    on='date'
)

In [17]:
merge_2_df = merge_1_df.merge(water_df,
               on='date',
               how='left'
               )

In [18]:
merge_2_df.head(2)

,date,STATION,NAME,LATITUDE,LONGITUDE,ELEVATION,DATE,ACMH,ACSH,AWND,...,WV01,YEAR,MONTH,DAY,TIME_PST,TIME_FLAG,SURF_TEMP_C,SURF_FLAG,BOT_TEMP_C,BOT_FLAG
0,1939-07-01,USW00023188,"SAN DIEGO INTERNATIONAL AIRPORT, CA US",32.7336,-117.1831,4.6,1939-07-01,NaN,NaN,NaN,...,NaN,1939,7,1,NaN,NaN,19.4,0.0,19.1,0.0
1,1939-07-02,USW00023188,"SAN DIEGO INTERNATIONAL AIRPORT, CA US",32.7336,-117.1831,4.6,1939-07-02,NaN,NaN,NaN,...,NaN,1939,7,2,NaN,NaN,19.8,0.0,19.5,0.0


#### 4. Curating the columns
##### Weather columns
There are many columns provided by the NOAA historical weather download. <br>
For details on all the available columns, please see the "weather-data-key.pdf" in the data folder.<br>

Listed below are the columns that were selected to keep in the dataframe.

date = Date of measurement
PRCP = Precipitation (mm or inches as per user preference, inches to hundredths on Daily Form pdf file) <br>
TMAX = Maximum temperature (Fahrenheit)<br>
TMIN = Minimum temperature (Fahrenheit)<br>
AWND = Average daily wind speed (meters per second or miles per hour as per user preference)<br>
<br>
BOT_TEMP_C = Water Temperature at the bottom of Scripp's pier <br>
SURF_TEMP_C = Water temperature at the surface (Target Variable)


In [27]:
final_colums = ['date','TMAX','TMIN','AWND', 'PRCP','BOT_TEMP_C', 'SURF_TEMP_C']

In [28]:
final_df = merge_2_df[final_colums].copy()

In [29]:
final_df.head(3)

,date,TMAX,TMIN,AWND,PRCP,BOT_TEMP_C,SURF_TEMP_C
0,1939-07-01,76.0,63.0,NaN,0.0,19.1,19.4
1,1939-07-02,74.0,65.0,NaN,0.0,19.5,19.8
2,1939-07-03,71.0,62.0,NaN,0.0,19.5,20.1


The data types are already in a desirable format so there is no need to set them.

In [31]:
final_df.dtypes

date           datetime64[us]
TMAX                  float64
TMIN                  float64
AWND                  float64
PRCP                  float64
BOT_TEMP_C            float64
SURF_TEMP_C           float64
dtype: object

### 5. Save merged data

In [30]:
final_df.to_parquet(r'..\data\raw\final-raw-merged-data.parquet',
                      engine='pyarrow',
                      index=False
                      )